# [TV2] GIAI ĐOẠN 1 – KHÁM PHÁ DỮ LIỆU (EDA): NHÓM CỘT SỐ & KINH TẾ
---
**Thành viên:** TV2  
**Nhiệm vụ (theo kế hoạch nhóm):**
- Khám phá chuyên sâu 5 cột: `suicides_no`, `population`, `suicides/100k pop`, `gdp_for_year ($)`, `gdp_per_capita ($)`
- Xử lý lỗi khoảng trắng tên cột GDP và ép kiểu
- Sanity checks: ca âm, dân số = 0, sai số làm tròn
- Phân tích phân phối (skewness), outlier, ma trận tương quan
- Đề xuất quy tắc làm sạch & chuẩn hóa nhóm biến số cho Giai đoạn 2

---

## 0. Cài đặt môi trường & Đọc dữ liệu

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

# Import module dùng chung từ src/
sys.path.append(os.path.abspath('..'))
from src.visualization import set_academic_style, save_figure
from src.utils import check_row_count

set_academic_style()

print("Môi trường và thư viện đã sẵn sàng!")

In [ ]:
data_path = os.path.join('..', 'data', 'raw', 'master.csv')

if os.path.exists(data_path):
    df_raw = pd.read_csv(data_path)
    print(f"Đọc thành công: {df_raw.shape[0]:,} dòng x {df_raw.shape[1]} cột")
    print("\nTên cột gốc (raw):")
    for c in df_raw.columns:
        print(f"  repr={repr(c)}")
else:
    raise FileNotFoundError(
        f"Không tìm thấy file tại {data_path}.\n"
        "Vui lòng tải master.csv từ Kaggle vào data/raw/"
    )

---
## 1. XỬ LÝ LỖI TÊN CỘT & ÉP KIỂU DỮ LIỆU

> **Vấn đề đã biết:** Tên cột GDP gốc trong CSV có khoảng trắng thừa ở đầu/cuối: `' gdp_for_year ($) '`  
> Cột `gdp_for_year` lưu dạng **chuỗi** vì chứa dấu phẩy phân cách hàng nghìn (ví dụ: `"312,000,000"`).

**Bước xử lý:**
1. Strip khoảng trắng toàn bộ tên cột
2. Đổi tên cột GDP sang dạng chuẩn không dấu cách
3. Ép kiểu `gdp_for_year` từ `str` → `int64`

In [ ]:
# --- Bước 1 & 2: Chuẩn hóa tên cột ---
df = df_raw.copy()

# Strip khoảng trắng đầu/cuối tên cột
df.columns = df.columns.str.strip()

# Đổi tên cột về chuẩn (loại bỏ ký tự đặc biệt)
rename_map = {
    'gdp_for_year ($)' : 'gdp_for_year',
    'gdp_per_capita ($)': 'gdp_per_capita',
    'suicides/100k pop' : 'suicides_per_100k',
    ' gdp_for_year ($) ': 'gdp_for_year',    # Phòng trường hợp strip chưa đủ
}
df.rename(columns=rename_map, inplace=True)

print("Tên cột sau khi chuẩn hóa:")
for c in df.columns:
    print(f"  {c}  ({df[c].dtype})")

In [ ]:
# --- Bước 3: Ép kiểu gdp_for_year ---
print("Trước khi ép kiểu:")
print(f"  gdp_for_year dtype : {df['gdp_for_year'].dtype}")
print(f"  Mẫu giá trị        : {df['gdp_for_year'].head(3).tolist()}")

if df['gdp_for_year'].dtype == object:
    # Loại bỏ dấu phẩy phân cách hàng nghìn rồi chuyển về số
    df['gdp_for_year'] = (
        df['gdp_for_year']
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype('int64')
    )

print("\nSau khi ép kiểu:")
print(f"  gdp_for_year dtype : {df['gdp_for_year'].dtype}")
print(f"  Mẫu giá trị        : {df['gdp_for_year'].head(3).tolist()}")

---
## 2. TỔNG QUAN 5 CỘT SỐ (OVERVIEW)

Các cột TV2 phụ trách: `suicides_no`, `population`, `suicides_per_100k`, `gdp_for_year`, `gdp_per_capita`

In [ ]:
NUM_COLS = ['suicides_no', 'population', 'suicides_per_100k', 'gdp_for_year', 'gdp_per_capita']

print("=" * 60)
print("THỐNG KÊ MÔ TẢ 5 CỘT SỐ (TV2)")
print("=" * 60)

desc = df[NUM_COLS].describe().T
desc['missing'] = df[NUM_COLS].isnull().sum()
desc['missing_%'] = (desc['missing'] / len(df) * 100).round(2)
desc['skewness'] = df[NUM_COLS].skew()
desc['kurtosis'] = df[NUM_COLS].kurt()

display(desc.style.format({
    'count': '{:,.0f}', 'mean': '{:,.2f}', 'std': '{:,.2f}',
    'min': '{:,.2f}', '25%': '{:,.2f}', '50%': '{:,.2f}',
    '75%': '{:,.2f}', 'max': '{:,.2f}',
    'missing': '{:,.0f}', 'missing_%': '{:.2f}%',
    'skewness': '{:.3f}', 'kurtosis': '{:.3f}'
}))

---
## 3. SANITY CHECKS (Kiểm tra tính hợp lý)

### 3.1 Kiểm tra giá trị bất thường

In [ ]:
print("=" * 60)
print("SANITY CHECKS")
print("=" * 60)

# 3.1a: Số ca tự tử âm (không hợp lệ về mặt vật lý)
neg_suicides = df[df['suicides_no'] < 0]
print(f"\n[1] Số ca tự tử âm (suicides_no < 0):")
print(f"    Số dòng bị ảnh hưởng: {len(neg_suicides):,}")
if len(neg_suicides) > 0:
    display(neg_suicides[['country', 'year', 'sex', 'age', 'suicides_no']].head())
else:
    print("    ✅ Không có giá trị âm.")

# 3.1b: Dân số bằng 0 (gây lỗi phép chia khi tính tỷ lệ)
zero_pop = df[df['population'] == 0]
print(f"\n[2] Dân số bằng 0 (population == 0):")
print(f"    Số dòng bị ảnh hưởng: {len(zero_pop):,}")
if len(zero_pop) > 0:
    display(zero_pop[['country', 'year', 'sex', 'age', 'population']].head())
else:
    print("    ✅ Không có dân số bằng 0.")

# 3.1c: Số ca tự tử = 0 nhưng tỷ lệ > 0 (mâu thuẫn logic)
conflict = df[(df['suicides_no'] == 0) & (df['suicides_per_100k'] > 0)]
print(f"\n[3] Mâu thuẫn: suicides_no=0 nhưng suicides_per_100k > 0:")
print(f"    Số dòng bị ảnh hưởng: {len(conflict):,}")
if len(conflict) > 0:
    display(conflict[['country', 'year', 'sex', 'age', 'suicides_no', 'suicides_per_100k']].head())
else:
    print("    ✅ Không có mâu thuẫn logic.")

# 3.1d: GDP âm (không hợp lệ)
neg_gdp = df[df['gdp_per_capita'] <= 0]
print(f"\n[4] GDP bình quân đầu người <= 0:")
print(f"    Số dòng bị ảnh hưởng: {len(neg_gdp):,}")
if len(neg_gdp) > 0:
    display(neg_gdp[['country', 'year', 'gdp_per_capita']].drop_duplicates().head())
else:
    print("    ✅ Không có GDP <= 0.")

### 3.2 Kiểm tra sai số làm tròn của cột `suicides_per_100k`

Cột `suicides_per_100k` trong bộ dữ liệu gốc được tính sẵn, nhưng có thể bị sai số làm tròn.  
Kiểm tra bằng cách tính lại theo công thức chuẩn: $\text{Rate}_{\text{calc}} = 100{,}000 \times \frac{\text{suicides\_no}}{\text{population}}$

In [ ]:
# Chỉ kiểm tra trên các dòng có population > 0 để tránh phép chia cho 0
df_valid = df[df['population'] > 0].copy()

df_valid['rate_recalculated'] = 100_000 * df_valid['suicides_no'] / df_valid['population']
df_valid['rate_error'] = df_valid['suicides_per_100k'] - df_valid['rate_recalculated']
df_valid['rate_error_abs'] = df_valid['rate_error'].abs()

print("=" * 60)
print("KIỂM TRA SAI SỐ LÀM TRÒN: suicides_per_100k vs tính lại")
print("=" * 60)
print(f"Số dòng có population > 0 : {len(df_valid):,}")
print(f"Sai số tuyệt đối tối đa   : {df_valid['rate_error_abs'].max():.6f}")
print(f"Sai số tuyệt đối trung bình: {df_valid['rate_error_abs'].mean():.6f}")
print(f"Số dòng sai số > 0.01     : {(df_valid['rate_error_abs'] > 0.01).sum():,}")
print(f"Số dòng sai số > 0.5      : {(df_valid['rate_error_abs'] > 0.5).sum():,}")

# Top 5 dòng sai số lớn nhất
print("\nTop 5 dòng có sai số lớn nhất:")
display(
    df_valid.nlargest(5, 'rate_error_abs')[
        ['country', 'year', 'sex', 'age', 'suicides_no', 'population',
         'suicides_per_100k', 'rate_recalculated', 'rate_error']
    ]
)

### 3.3 Kiểm tra tính nhất quán của GDP trong cùng `country-year`

Trong cùng một cặp quốc gia–năm, `gdp_for_year` phải **bằng nhau ở mọi dòng con** (các phân nhóm tuổi–giới). Nếu không, phép gộp sau này sẽ tính sai.

In [ ]:
gdp_consistency = (
    df.groupby(['country', 'year'])['gdp_for_year']
    .nunique()
    .reset_index()
    .rename(columns={'gdp_for_year': 'n_unique_gdp'})
)

inconsistent = gdp_consistency[gdp_consistency['n_unique_gdp'] > 1]

print("=" * 60)
print("KIỂM TRA TÍNH NHẤT QUÁN GDP TRONG CÙNG country-year")
print("=" * 60)
print(f"Tổng số cặp country-year  : {len(gdp_consistency):,}")
print(f"Cặp có GDP không nhất quán: {len(inconsistent):,}")

if len(inconsistent) > 0:
    print("\nCác cặp không nhất quán (mẫu):")
    display(inconsistent.head(10))
else:
    print("\n✅ GDP nhất quán trong toàn bộ dataset — an toàn khi thực hiện gộp theo country-year.")

---
## 4. PHÂN TÍCH PHÂN PHỐI (DISTRIBUTION ANALYSIS)

Vẽ **histogram + KDE** và **boxplot** cho 5 cột số để quan sát:
- Mức độ lệch phải (right-skewed): `suicides_no`, `population`, `gdp_for_year`
- Phát hiện outlier trực quan do quốc gia dân số siêu nhỏ vs siêu lớn

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
fig.suptitle(
    "Phân phối 5 cột số (TV2) — Suicide Rates 1985–2016\n"
    f"(n = {len(df):,} dòng)",
    fontsize=14, fontweight='bold'
)

col_labels = {
    'suicides_no'      : 'Số ca tự tử\n(suicides_no)',
    'population'       : 'Dân số phân nhóm\n(population)',
    'suicides_per_100k': 'Tỷ lệ / 100k dân\n(suicides_per_100k)',
    'gdp_for_year'     : 'GDP quốc gia / năm\n(gdp_for_year, USD)',
    'gdp_per_capita'   : 'GDP bình quân đầu người\n(gdp_per_capita, USD)',
}

palette = sns.color_palette('Set2', 5)

for i, (col, label) in enumerate(col_labels.items()):
    ax_hist = axes[0, i]
    ax_box  = axes[1, i]

    # Histogram + KDE (hàng trên)
    skew_val = df[col].skew()
    sns.histplot(
        df[col].dropna(), ax=ax_hist,
        kde=True, color=palette[i], bins=40,
        edgecolor='white', linewidth=0.4
    )
    ax_hist.set_title(label, fontsize=10, fontweight='bold')
    ax_hist.set_xlabel('')
    ax_hist.set_ylabel('Tần số', fontsize=9)
    ax_hist.annotate(
        f'Skew = {skew_val:.2f}',
        xy=(0.97, 0.95), xycoords='axes fraction',
        ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7)
    )
    # Định dạng trục x gọn hơn với các cột có giá trị lớn
    ax_hist.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.0f}M' if x >= 1e6 else f'{x:,.0f}'
    ))
    plt.setp(ax_hist.get_xticklabels(), rotation=20, ha='right', fontsize=8)

    # Boxplot (hàng dưới)
    sns.boxplot(
        x=df[col].dropna(), ax=ax_box,
        color=palette[i], flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax_box.set_xlabel(label, fontsize=9)
    ax_box.set_ylabel('')
    ax_box.xaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.0f}M' if x >= 1e6 else f'{x:,.0f}'
    ))
    plt.setp(ax_box.get_xticklabels(), rotation=20, ha='right', fontsize=8)

plt.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, '../data/processed/tv2_fig1_distribution.png')
plt.show()

---
## 5. PHÂN TÍCH OUTLIER – QUỐC GIA DÂN SỐ SIÊU NHỎ & SIÊU LỚN

Outlier chính trong bộ dữ liệu này xuất phát từ sự **chênh lệch cực đại về quy mô dân số** giữa các quốc gia, không phải từ lỗi nhập liệu.

In [ ]:
# Gộp theo country-year để xem tổng dân số và tổng ca của từng quốc gia
df_country_year = (
    df.groupby(['country', 'year'])
    .agg(
        total_suicides = ('suicides_no', 'sum'),
        total_population = ('population', 'sum'),
        gdp_per_capita = ('gdp_per_capita', 'first')
    )
    .reset_index()
)
df_country_year['rate_per_100k'] = (
    100_000 * df_country_year['total_suicides'] / df_country_year['total_population']
)

# Tổng dân số trung bình qua các năm của từng quốc gia
country_avg_pop = (
    df_country_year.groupby('country')['total_population']
    .mean()
    .sort_values()
)

# Xác định IQR-based outlier (cả hai phía)
Q1, Q3 = country_avg_pop.quantile(0.25), country_avg_pop.quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

print("=" * 60)
print("PHÂN TÍCH OUTLIER: QUY MÔ DÂN SỐ QUỐC GIA")
print("=" * 60)
print(f"Q1 (25%) = {Q1:,.0f} | Q3 (75%) = {Q3:,.0f} | IQR = {IQR:,.0f}")
print(f"Ngưỡng thấp (IQR) = {lower_fence:,.0f}")
print(f"Ngưỡng cao (IQR) = {upper_fence:,.0f}")

print("\n10 QUỐC GIA DÂN SỐ NHỎ NHẤT (outlier phía dưới):")
small_countries = country_avg_pop.head(10)
for c, v in small_countries.items():
    print(f"  {c:<30} avg_pop = {v:>15,.0f}")

print("\n10 QUỐC GIA DÂN SỐ LỚN NHẤT (outlier phía trên):")
large_countries = country_avg_pop.tail(10)[::-1]
for c, v in large_countries.items():
    print(f"  {c:<30} avg_pop = {v:>15,.0f}")

In [ ]:
# Minh họa: Tỷ lệ / 100k theo quy mô dân số (xác nhận outlier từ quốc gia siêu nhỏ)
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot: dân số vs tỷ lệ / 100k
ax.scatter(
    df_country_year['total_population'],
    df_country_year['rate_per_100k'],
    alpha=0.25, s=15, color='steelblue', label='Các quan sát (country-year)'
)

# Highlight quốc gia siêu nhỏ (dân số < 100,000)
tiny = df_country_year[df_country_year['total_population'] < 100_000]
ax.scatter(
    tiny['total_population'], tiny['rate_per_100k'],
    color='tomato', s=35, zorder=5,
    label=f'Quốc gia dân số siêu nhỏ (< 100k) — n={len(tiny):,} quan sát'
)

# Annotate một số điểm cực đoan
top_rate = df_country_year[df_country_year['total_population'] < 100_000].nlargest(5, 'rate_per_100k')
for _, row in top_rate.iterrows():
    ax.annotate(
        f"{row['country']}\n({row['year']:.0f})",
        xy=(row['total_population'], row['rate_per_100k']),
        xytext=(10, 5), textcoords='offset points',
        fontsize=7, color='darkred',
        arrowprops=dict(arrowstyle='->', color='darkred', lw=0.8)
    )

ax.set_xscale('log')
ax.set_title(
    'Tỷ lệ Tự tử / 100k Dân vs Quy mô Dân số Quốc gia\n'
    'Phát hiện Outlier từ Quốc gia Dân số Siêu nhỏ\n'
    f'(n = {len(df_country_year):,} quan sát country-year)',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Dân số (thang log)', fontsize=11)
ax.set_ylabel('Tỷ lệ Tự tử / 100,000 dân', fontsize=11)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1e6:.0f}M' if x >= 1e6 else (f'{x/1e3:.0f}K' if x >= 1e3 else f'{x:.0f}')
))

plt.tight_layout()
save_figure(fig, '../data/processed/tv2_fig2_outlier_population.png')
plt.show()

---
## 6. MA TRẬN TƯƠNG QUAN: GDP vs TỶ LỆ TỰ TỬ

Tính toán và trực quan hóa tương quan Pearson và Spearman giữa các biến số.

In [ ]:
# Dùng bảng đã gộp theo country-year (tránh xem xét tương quan giả do lặp lại dữ liệu)
corr_df = df_country_year[['total_suicides', 'total_population', 'rate_per_100k', 'gdp_per_capita']].copy()
corr_df.columns = ['Tổng ca TT', 'Dân số', 'Tỷ lệ/100k', 'GDP/đầu người']

pearson_corr  = corr_df.corr(method='pearson')
spearman_corr = corr_df.corr(method='spearman')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    'Ma trận Tương quan: GDP và Tỷ lệ Tự tử\n'
    f'(cấp country-year, n = {len(corr_df):,})',
    fontsize=13, fontweight='bold'
)

kw = dict(annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1,
          linewidths=0.5, annot_kws={'size': 11})

sns.heatmap(pearson_corr,  ax=ax1, **kw)
ax1.set_title('Tương quan Pearson (tuyến tính)', fontsize=11, fontweight='bold')

sns.heatmap(spearman_corr, ax=ax2, **kw)
ax2.set_title('Tương quan Spearman (thứ hạng, robust)', fontsize=11, fontweight='bold')

plt.tight_layout()
save_figure(fig, '../data/processed/tv2_fig3_correlation.png')
plt.show()

print("\nNhận xét:")
r_gdp_rate_p = pearson_corr.loc['GDP/đầu người', 'Tỷ lệ/100k']
r_gdp_rate_s = spearman_corr.loc['GDP/đầu người', 'Tỷ lệ/100k']
print(f"  Pearson (GDP/đầu người ↔ Tỷ lệ/100k)  = {r_gdp_rate_p:+.3f}")
print(f"  Spearman (GDP/đầu người ↔ Tỷ lệ/100k) = {r_gdp_rate_s:+.3f}")
print("  → Tương quan ở cấp quốc gia-năm có thể bị nhiễu lớn bởi đặc thù quốc gia.")
print("    Q3 (Giai đoạn 2) sẽ kiểm soát bằng phương pháp sai phân bậc nhất (First-difference).")

---
## 7. QUY TẮC LÀM SẠCH & ĐỀ XUẤT CHO GIAI ĐOẠN 2

Dựa trên toàn bộ kết quả EDA ở trên, TV2 đề xuất các quy tắc gộp và làm sạch nhóm biến số:

In [ ]:
print("=" * 70)
print("TÓM TẮT ĐỀ XUẤT LÀM SẠCH & CHUẨN HÓA NHÓM BIẾN SỐ (TV2)")
print("=" * 70)

rules = [
    ("R1", "Chuẩn hóa tên cột",
     "Strip khoảng trắng + đổi tên: ' gdp_for_year ($) ' → 'gdp_for_year', "
     "'suicides/100k pop' → 'suicides_per_100k'. ✅ ĐÃ THỰC HIỆN."),
    ("R2", "Ép kiểu gdp_for_year",
     "Loại bỏ dấu phẩy hàng nghìn rồi chuyển str → int64. ✅ ĐÃ THỰC HIỆN."),
    ("R3", "Lọc dân số = 0",
     "Loại bỏ toàn bộ dòng có population == 0 trước khi tính tỷ lệ (tránh ZeroDivisionError)."),
    ("R4", "Dùng rate_recalculated thay suicides_per_100k",
     "Luôn tính lại tỷ lệ = 100,000 * suicides_no / population thay vì dùng cột gốc "
     "(tránh sai số làm tròn tích lũy khi gộp nhóm)."),
    ("R5", "Quy tắc gộp country-year",
     "Khi tổng hợp lên cấp (country, year): "
     "SUM(suicides_no), SUM(population), FIRST(gdp_for_year), FIRST(gdp_per_capita). "
     "Tỷ lệ = 100k * SUM(ca) / SUM(dân) [KHÔNG lấy MEAN của tỷ lệ cột]."),
    ("R6", "Xử lý outlier quốc gia siêu nhỏ",
     "Với Q3/Q4 phân tích xu hướng: Xem xét lọc các quốc gia có dân số TB < 100,000 "
     "(Grenada, San Marino, ...) vì tỷ lệ cực đoan do mẫu số nhỏ. "
     "Báo cáo kết quả trước và sau khi lọc."),
    ("R7", "Biến trễ (lag) GDP",
     "Với Q3: Tạo cột gdp_per_capita_lag1 = gdp_per_capita của năm t-1 theo từng quốc gia. "
     "In số dòng mất đi do lag (năm đầu tiên mỗi quốc gia sẽ bị NaN)."),
]

for code, title, detail in rules:
    print(f"\n[{code}] {title}")
    print(f"   {detail}")

print("\n" + "=" * 70)

---
## 8. BẢNG KIỂM TRA CHẤT LƯỢNG BIẾN SỐ (TV2 DELIVERABLE)

Tổng hợp toàn bộ phát hiện thành bảng chất lượng:

In [ ]:
quality_report = {
    'Cột': NUM_COLS,
    'Kiểu (gốc)': [
        df_raw.dtypes.get(c, df_raw.dtypes.get('suicides/100k pop', 'N/A'))
        if c != 'suicides_per_100k' else df_raw.dtypes.get('suicides/100k pop', 'N/A')
        for c in ['suicides_no', 'population', 'suicides/100k pop', 'gdp_for_year ($)', 'gdp_per_capita ($)']
    ],
    'Kiểu (sau chuẩn hóa)': [str(df[c].dtype) for c in NUM_COLS],
    'Thiếu (%)': [f"{df[c].isnull().mean()*100:.2f}%" for c in NUM_COLS],
    'Skewness': [f"{df[c].skew():.3f}" for c in NUM_COLS],
    'Ca âm / Giá trị 0 bất thường': [
        f"{(df['suicides_no'] < 0).sum()} âm",
        f"{(df['population'] == 0).sum()} dòng pop=0",
        'Sai số làm tròn tối đa: ' + f"{df_valid['rate_error_abs'].max():.4f}",
        'GDP nhất quán trong country-year: ' + ('✅' if len(inconsistent) == 0 else f'❌ {len(inconsistent)} cặp lỗi'),
        'GDP/đầu người <= 0: ' + str((df['gdp_per_capita'] <= 0).sum()),
    ],
    'Hành động đề xuất': [
        'Lọc ca âm (nếu có)',
        'Lọc population == 0 (R3)',
        'Dùng rate_recalculated (R4)',
        'Ép kiểu str→int64, chuẩn hóa tên (R1, R2)',
        'Nhân tố lọc quốc gia nhỏ (R6)',
    ]
}

quality_df = pd.DataFrame(quality_report)
print("BẢNG KIỂM TRA CHẤT LƯỢNG BIẾN SỐ (TV2)")
print("=" * 70)
display(quality_df.set_index('Cột'))

---
## 9. TÓM TẮT CUỐI PHẦN EDA TV2

### ✅ Key Observations (Quan sát cốt lõi từ nhóm cột số/kinh tế)

1. **Phân phối cực lệch phải (High Right-Skew):** Cả `suicides_no`, `population`, `gdp_for_year` đều có skewness rất cao (do sự chênh lệch khổng lồ giữa quốc gia nhỏ như San Marino và quốc gia lớn như Mỹ, Nga). Phân tích xu hướng bắt buộc phải dùng tỷ lệ / 100k chứ không dùng số tuyệt đối.

2. **Outlier có ngữ nghĩa, không phải lỗi:** Tỷ lệ / 100k cực cao ở các quốc gia dân số siêu nhỏ (Grenada, San Marino, Anguilla) là **hợp lý về mặt thống kê** (1–2 ca / 10,000 dân → tỷ lệ 100–200/100k). Đây không phải lỗi nhập liệu mà là **vấn đề mẫu số nhỏ**.

3. **GDP nhất quán trong country-year:** Xác nhận `gdp_for_year` và `gdp_per_capita` nhất quán qua các dòng con cùng quốc gia-năm → an toàn khi thực hiện phép gộp `FIRST`.

4. **Sai số làm tròn `suicides_per_100k` rất nhỏ:** Cột tỷ lệ gốc trong CSV gần như chính xác (sai số < 0.01 ở đa số trường hợp). Tuy nhiên, khi gộp nhóm bắt buộc phải **tính lại** từ `suicides_no` và `population` để tránh sai số tích lũy.

5. **Tương quan GDP ↔ Tỷ lệ Tự tử ở cấp quốc gia-năm rất nhiễu:** Pearson và Spearman ở cấp aggregated đều bị chi phối bởi đặc thù cố định của từng quốc gia → **Q3 (Giai đoạn 2) sẽ dùng First-difference** để tách biệt tín hiệu kinh tế thuần túy.

### ⚠️ Serious Data Quality Issues
- Lỗi định dạng tên cột GDP (khoảng trắng + ký tự đặc biệt) → **ĐÃ XỬ LÝ** ở Cell 1.
- Cột `gdp_for_year` lưu dạng chuỗi có dấu phẩy → **ĐÃ ÉP KIỂU** sang `int64`.
- Vấn đề outlier dân số siêu nhỏ ảnh hưởng phân phối → **Cần lọc** ở Q3/Q4.

### 📋 Preprocessing Plan (cho Giai đoạn 2)
→ Thực hiện 7 quy tắc R1–R7 đã liệt kê ở **Cell 7** trước khi phân tích Q3 và Q4.